# Task Discovery

Group IDE events into activity bursts, segment the activity stream, and cluster segments with similar behaviour. Process maps and context features help interpret the clusters as tasks.


## 1. Prepare activity features

### 1.1 From sessions to activity bursts

Start with the saved sessions. Normalize event context, merge compatible events into bursts, and build features for detecting changes in activity.


### 1.2 Configuration

The defaults use the full preprocessed dataset. Adjust `SESSION_SAMPLE_FRACTION` for a smaller exploratory run.


In [ ]:
from __future__ import annotations

from pathlib import Path
from collections import Counter
import gc
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=FutureWarning)

from pathlib import Path
import sys

_root = Path.cwd().resolve()
while not (_root / 'project_paths.py').is_file():
    if _root == _root.parent:
        raise RuntimeError('Open this notebook from within the ide-task-inference repository.')
    _root = _root.parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from project_paths import get_paths, export_discovery_results
paths = get_paths()

CODE_DIR = paths.root
from pipeline.process_mining.activity_abstraction import build_activity_bursts
from pipeline.process_mining.canonicalize import canonical_event_profile, canonicalize_session
from pipeline.process_mining.config import ProcessMiningConfig
from pipeline.process_mining.context_features import (
    activity_feature_dicts,
    cluster_feature_rankings,
    enrich_canonical_context,
    enrich_segments_with_activity_context,
    fit_reduce_feature_dicts,
    iter_activity_feature_dict_batches,
    segment_feature_dicts,
    transform_feature_dicts,
)
from pipeline.process_mining.discovery import discover_dfg, frequent_traces, mine_local_patterns, summarize_case_group
from pipeline.process_mining.fuzzy_miner import (
    FuzzyMinerConfig,
    discover_fuzzy_process,
    fuzzy_graph_tables,
    render_fuzzy_process_contact_sheet,
    render_fuzzy_process_graph,
)
from pipeline.process_mining.load_events import iter_session_event_lists
from pipeline.process_mining.representation_segmentation import (
    RepresentationSegmentationConfig,
    segment_activity_streams_by_representation,
    segment_summary_table,
)

MAX_SESSIONS = None
SESSION_STRIDE = 1
SESSION_OFFSET = 0
SESSION_SAMPLE_FRACTION = 1.0
SESSION_SAMPLE_SEED = 42
MIN_EVENTS_PER_SESSION = 5
RANDOM_SEED = 42

pm_config = ProcessMiningConfig()
pm_config.max_sessions = MAX_SESSIONS
pm_config.session_stride = SESSION_STRIDE
pm_config.session_offset = SESSION_OFFSET
pm_config.session_sample_fraction = SESSION_SAMPLE_FRACTION
pm_config.session_sample_seed = SESSION_SAMPLE_SEED
pm_config.min_events_per_session = MIN_EVENTS_PER_SESSION

segmentation_config = RepresentationSegmentationConfig(
    min_segment_activities=3,
    max_segment_activities=60,
    context_radius=6,
    boundary_quantile=0.94,
    min_boundary_distance=0.25,
    idle_gap_seconds=30 * 60,
)

ACTIVITY_COMPONENTS = 32
ACTIVITY_FIT_SAMPLE_ROWS = 150_000
FEATURE_TRANSFORM_BATCH_ROWS = 100_000
SEGMENT_COMPONENTS = 32
HDBSCAN_MIN_CLUSTER_SIZE = 300
HDBSCAN_MIN_SAMPLES = 10
HDBSCAN_TARGET_CLUSTERS = 25
HDBSCAN_MAX_CANDIDATES = 48
HDBSCAN_SEARCH_SAMPLE_ROWS = 40_000
HDBSCAN_SILHOUETTE_SAMPLE_ROWS = 10_000
HDBSCAN_FIT_SAMPLE_ROWS = 200_000
HDBSCAN_PREDICT_BATCH_ROWS = 50_000

OUTPUT_DIR = paths.discovery
RESULT_DIR = paths.discovery_results
FIGURE_DIR = RESULT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"CODE_DIR: {CODE_DIR}")
print(f"Input sessions: {pm_config.sessions_path}")
print(f"Output directory: {OUTPUT_DIR}")
print(pm_config)
print(segmentation_config)




### 1.3 Load and normalize events

Read `sessions.jsonl` and normalize context such as file role, command type, and navigation kind.


In [ ]:
CANONICAL_DISPLAY_SAMPLE_ROWS = 5_000

canonical_frames: list[pd.DataFrame | None] = []
session_event_counts: list[int] = []
canonical_sample_frames: list[pd.DataFrame] = []
canonical_sample_rows = 0

canonical_type_counts = Counter()
context_value_counts = {
    column: Counter()
    for column in [
        "action_type", "file_role", "file_extension", "command_family",
        "symbol_kind", "navigation_kind", "debug_action", "vcs_action",
    ]
}
command_other_total = 0
command_other_refined = 0
command_other_profile_counts = Counter()
loaded_sessions = 0
canonical_event_total = 0

for session_index, session_events in enumerate(iter_session_event_lists(pm_config)):
    canonical = canonicalize_session(session_events, session_index=session_index)
    if canonical.empty:
        continue
    canonical = enrich_canonical_context(canonical)

    # Keep per-session frames for section 1.5, but avoid also materializing one
    # full concatenated canonical_events table. The display dataframe below is
    # only a small sample; aggregate counters keep the full-data profile.
    canonical_frames.append(canonical)
    session_event_counts.append(len(canonical))
    loaded_sessions += 1
    canonical_event_total += len(canonical)
    canonical_type_counts.update(canonical["canonical_type"].astype(str).tolist())

    for column, counter in context_value_counts.items():
        if column in canonical.columns:
            counter.update(canonical[column].fillna("").astype(str).tolist())

    command_other = canonical.loc[canonical["canonical_type"] == "command_other"]
    if not command_other.empty:
        command_other_total += len(command_other)
        command_other_refined += int((command_other["action_type"] != "command_other").sum())
        for row in command_other[["action_type", "command_family", "command_category", "command_name"]].fillna("").itertuples(index=False):
            command_other_profile_counts.update([tuple(str(value) for value in row)])

    if canonical_sample_rows < CANONICAL_DISPLAY_SAMPLE_ROWS:
        take = min(CANONICAL_DISPLAY_SAMPLE_ROWS - canonical_sample_rows, len(canonical))
        canonical_sample_frames.append(canonical.head(take).copy())
        canonical_sample_rows += take

canonical_events = pd.concat(canonical_sample_frames, ignore_index=True) if canonical_sample_frames else pd.DataFrame()
profile = {
    "events": int(canonical_event_total),
    "sessions": int(loaded_sessions),
    "top_event_types": canonical_type_counts.most_common(15),
}

print("Canonical event profile:")
print(profile)
print(f"Loaded sessions: {loaded_sessions:,}")
print(f"Loaded canonical events: {canonical_event_total:,}")
print(f"Canonical display sample retained in memory: {len(canonical_events):,} rows")

if not canonical_events.empty:
    display(canonical_events.head(5))

for column, counter in context_value_counts.items():
    print(f"\n{column}")
    print(pd.Series(dict(counter.most_common(10)), name="count").to_string())


### 1.4 Refine generic commands

Use command names and categories to refine `command_other` events. Keep those fields available for feature extraction.


In [ ]:
if canonical_event_total == 0:
    print("No canonical events loaded.")
else:
    print(f"command_other events: {command_other_total:,} / {canonical_event_total:,}")
    if command_other_total:
        print(
            "command_other refined into non-generic action types: "
            f"{command_other_refined:,} ({command_other_refined / command_other_total:.1%})"
        )
        command_other_rows = [
            {
                "action_type": key[0],
                "command_family": key[1],
                "command_category": key[2],
                "command_name": key[3],
                "count": int(count),
            }
            for key, count in command_other_profile_counts.most_common(25)
        ]
        display(pd.DataFrame(command_other_rows))


### 1.5 Build activity bursts

Merge consecutive compatible events and summarize their context in each burst.


In [ ]:
activity_frames: list[pd.DataFrame] = []
activity_total = 0

# Section 1.3 kept only per-session canonical frames, not a second full concat.
# Here each canonical frame is consumed once and immediately released, so the
# memory peak is not full canonical data plus full activity data at the same time.
for frame_index, canonical in enumerate(canonical_frames):
    if canonical is None or canonical.empty:
        canonical_frames[frame_index] = None
        continue

    activities = build_activity_bursts(canonical, pm_config)
    canonical_frames[frame_index] = None

    if not activities.empty:
        activity_total += len(activities)
        activity_frames.append(activities)

    if (frame_index + 1) % 250 == 0:
        print(f"Processed {frame_index + 1:,} canonical session frames; activities so far: {activity_total:,}")
        gc.collect()

# Drop the now-empty holder before concatenating activity frames. pd.concat can
# briefly duplicate activity data, so freeing canonical frames first matters.
del canonical_frames
gc.collect()

activity_bursts = pd.concat(activity_frames, ignore_index=True, copy=False) if activity_frames else pd.DataFrame()
del activity_frames
gc.collect()

if not activity_bursts.empty:
    activity_bursts = activity_bursts.sort_values(["session_index", "start_time", "activity_index"]).reset_index(drop=True)

print(f"Canonical events: {canonical_event_total:,}")
print(f"Activity bursts: {len(activity_bursts):,}")
if len(activity_bursts) > 0:
    print(f"Compression ratio: {canonical_event_total / len(activity_bursts):.2f} events per activity")
    display(activity_bursts.head(8))
    display(activity_bursts["activity"].value_counts().rename("count").to_frame())


### 1.6 Reduce activity features

Fit the feature transform on a bounded sample, then apply it to all bursts in batches to limit memory use.


In [ ]:
if activity_bursts.empty:
    activity_feature_pool = []
    activity_reduction = fit_reduce_feature_dicts([], max_components=ACTIVITY_COMPONENTS, random_state=RANDOM_SEED)
    activity_embeddings = np.empty((0, 0), dtype=np.float32)
else:
    activity_count = len(activity_bursts)
    if activity_count <= ACTIVITY_FIT_SAMPLE_ROWS:
        fit_activities = activity_bursts
        fit_session_count = int(activity_bursts["session_index"].nunique())
    else:
        session_sizes = activity_bursts.groupby("session_index", sort=False).size()
        shuffled_sessions = np.array(session_sizes.index.tolist())
        rng = np.random.default_rng(RANDOM_SEED)
        rng.shuffle(shuffled_sessions)

        selected_sessions = []
        selected_rows = 0
        for session_index in shuffled_sessions:
            selected_sessions.append(session_index)
            selected_rows += int(session_sizes.loc[session_index])
            if selected_rows >= ACTIVITY_FIT_SAMPLE_ROWS:
                break

        fit_activities = activity_bursts.loc[activity_bursts["session_index"].isin(selected_sessions)]
        fit_session_count = len(selected_sessions)

    print(
        "Fitting activity representation on "
        f"{len(fit_activities):,}/{activity_count:,} activity rows "
        f"from {fit_session_count:,} sessions"
    )
    activity_feature_pool = activity_feature_dicts(fit_activities)
    activity_reduction = fit_reduce_feature_dicts(
        activity_feature_pool,
        max_components=ACTIVITY_COMPONENTS,
        random_state=RANDOM_SEED,
    )
    del activity_feature_pool
    if fit_activities is not activity_bursts:
        del fit_activities
    gc.collect()

    activity_embeddings = np.empty((activity_count, activity_reduction.n_components), dtype=np.float32)
    row_offset = 0
    for batch_index, feature_batch in enumerate(
        iter_activity_feature_dict_batches(
            activity_bursts,
            batch_size=FEATURE_TRANSFORM_BATCH_ROWS,
            sort=False,
        ),
        start=1,
    ):
        transformed = transform_feature_dicts(feature_batch, activity_reduction)
        batch_len = len(feature_batch)
        activity_embeddings[row_offset : row_offset + batch_len] = transformed
        row_offset += batch_len

        if batch_index == 1 or batch_index % 10 == 0:
            print(f"Transformed {row_offset:,}/{activity_count:,} activity rows")
        del feature_batch, transformed
        gc.collect()

    if row_offset != activity_count:
        raise RuntimeError(f"Activity embedding row mismatch: {row_offset:,} != {activity_count:,}")

print(f"Activity feature rows transformed: {len(activity_bursts):,}")
print(f"Selected activity features: {len(activity_reduction.feature_names):,}")
print(f"Activity representation shape: {activity_embeddings.shape}")
print("Sample selected features:")
print(activity_reduction.feature_names[:30])


## 2. Adaptive segmentation

### 2.1 Detect changes in activity

Compare the activity representations on either side of a candidate boundary. Large changes suggest a split. The rules also account for inactivity, version control, segment length, and short fragments.


In [ ]:
segments, boundary_table, boundary_threshold = segment_activity_streams_by_representation(
    activity_bursts,
    activity_embeddings,
    config=segmentation_config,
)
segments = enrich_segments_with_activity_context(segments, activity_bursts)
segment_table = segment_summary_table(segments)

candidate_boundary_count = int(boundary_table.attrs.get("candidate_count", len(boundary_table)))
accepted_boundary_count = int(
    boundary_table.attrs.get(
        "accepted_count",
        int(boundary_table["split"].sum()) if not boundary_table.empty and "split" in boundary_table else 0,
    )
)
diagnostic_boundary_count = len(boundary_table)

cap_reason = f"max_segment_activities={segmentation_config.max_segment_activities}"
short_segment_ratio = float((segment_table["activity_count"] < segmentation_config.min_segment_activities).mean()) if not segment_table.empty else 0.0
cap_segment_ratio = float((segment_table["close_reason"] == cap_reason).mean()) if not segment_table.empty else 0.0
session_segment_counts = segment_table.groupby("session_index").size().sort_values(ascending=False) if not segment_table.empty else pd.Series(dtype=int)
top10_session_segment_share = float(session_segment_counts.head(10).sum() / len(segment_table)) if len(segment_table) else 0.0

print(f"Segments produced: {len(segments):,}")
print(f"Candidate boundaries scored: {candidate_boundary_count:,}")
print(f"Diagnostic boundary rows retained: {diagnostic_boundary_count:,}")
print(f"Accepted boundaries: {accepted_boundary_count:,}")
print(f"Representation boundary threshold: {boundary_threshold:.4f}")
print(f"Short fragment ratio (< min_segment_activities): {short_segment_ratio:.1%}")
print(f"Max-length guard ratio: {cap_segment_ratio:.1%}")
print(f"Top-10 session segment share: {top10_session_segment_share:.1%}")

if not segment_table.empty:
    display(segment_table.head(10))
    display(segment_table["close_reason"].value_counts().rename("count").to_frame())
if not boundary_table.empty:
    display(boundary_table.sort_values("representation_distance", ascending=False).head(10))


### 2.2 Segmentation diagnostics


In [ ]:
if not boundary_table.empty:
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.hist(boundary_table["representation_distance"], bins=40, color="#4C78A8", alpha=0.85)
    ax.axvline(boundary_threshold, color="#E45756", linewidth=2, label="threshold")
    ax.set_title("Representation-shift boundary distances")
    ax.set_xlabel("Cosine distance between local reduced contexts")
    ax.set_ylabel("Candidate boundary count")
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "representation_boundary_distance.png", dpi=160)
    plt.show()

if not segment_table.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].hist(segment_table["activity_count"], bins=range(1, int(segment_table["activity_count"].max()) + 2), color="#59A14F", alpha=0.85)
    axes[0].axvline(segmentation_config.min_segment_activities, color="#E45756", linestyle="--", linewidth=1.5, label="min")
    axes[0].axvline(segmentation_config.max_segment_activities, color="#B07AA1", linestyle="--", linewidth=1.5, label="max guard")
    axes[0].set_title("Segment length by activity count")
    axes[0].set_xlabel("Activities per segment")
    axes[0].set_ylabel("Segment count")
    axes[0].legend()

    duration_minutes = segment_table["duration_seconds"] / 60.0
    axes[1].hist(duration_minutes.clip(upper=120), bins=30, color="#F28E2B", alpha=0.85)
    axes[1].set_title("Segment duration distribution")
    axes[1].set_xlabel("Duration minutes, clipped at 120")
    axes[1].set_ylabel("Segment count")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "context_segment_length_duration.png", dpi=160)
    plt.show()

    top_sessions = session_segment_counts.head(15).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, max(3, 0.35 * len(top_sessions))))
    top_sessions.plot(kind="barh", ax=ax, color="#76B7B2")
    ax.set_title("Top sessions by produced segment count")
    ax.set_xlabel("Segment count")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "top_sessions_by_segment_count.png", dpi=160)
    plt.show()

    quality_rows = pd.DataFrame(
        [
            {"metric": "short_fragment_ratio", "value": short_segment_ratio},
            {"metric": "max_length_guard_ratio", "value": cap_segment_ratio},
            {"metric": "top10_session_segment_share", "value": top10_session_segment_share},
            {"metric": "median_segment_activities", "value": float(segment_table["activity_count"].median())},
            {"metric": "median_segment_duration_seconds", "value": float(segment_table["duration_seconds"].median())},
        ]
    )
    display(quality_rows)


## 3. Segment clustering


### 3.1 Build segment features

Describe each segment using its activity mix, sequence, and context. Filter features by variance and reduce the representation with TruncatedSVD.


In [ ]:
segment_feature_pool = segment_feature_dicts(segments)
segment_reduction = fit_reduce_feature_dicts(
    segment_feature_pool,
    max_components=SEGMENT_COMPONENTS,
    random_state=RANDOM_SEED,
)
segment_embeddings = segment_reduction.reduced

print(f"Segment feature rows: {len(segment_feature_pool):,}")
print(f"Selected segment features: {len(segment_reduction.feature_names):,}")
print(f"Segment representation shape: {segment_embeddings.shape}")
print("Sample selected segment features:")
print(segment_reduction.feature_names[:40])

if segment_reduction.reducer is not None:
    explained = pd.DataFrame(
        {
            "component": np.arange(1, segment_reduction.reducer.n_components + 1),
            "explained_variance_ratio": segment_reduction.reducer.explained_variance_ratio_,
            "cumulative_explained_variance_ratio": np.cumsum(segment_reduction.reducer.explained_variance_ratio_),
        }
    )
    display(explained.head(12))

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(
        explained["component"],
        explained["cumulative_explained_variance_ratio"],
        marker="o",
        linewidth=1.5,
        color="#4C78A8",
    )
    ax.set_title("Cumulative variance captured by segment SVD representation")
    ax.set_xlabel("SVD component")
    ax.set_ylabel("Cumulative explained variance ratio")
    ax.set_ylim(0, min(1.0, max(0.1, explained["cumulative_explained_variance_ratio"].max() * 1.1)))
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "segment_svd_explained_variance.png", dpi=160)
    plt.show()


### 3.1.1 PCA diagnostic view

Inspect a sample in PCA space. HDBSCAN uses the SVD representation.


In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

PCA_DIAGNOSTIC_SAMPLE_ROWS = 20_000
segment_pca_embeddings = np.empty((len(segment_feature_pool), 0), dtype=np.float32)
segment_pca_model = None

if segment_reduction.feature_matrix.shape[0] >= 3 and segment_reduction.feature_matrix.shape[1] >= 2:
    # PCA is a diagnostic mirror of the old notebook's PC plots. For full-data
    # runs we sample rows before densifying the sparse feature matrix; otherwise
    # the diagnostic itself can exceed memory even though clustering uses SVD.
    pca_rows = segment_reduction.feature_matrix.shape[0]
    if pca_rows > PCA_DIAGNOSTIC_SAMPLE_ROWS:
        rng = np.random.default_rng(RANDOM_SEED)
        pca_sample_indices = np.sort(rng.choice(pca_rows, size=PCA_DIAGNOSTIC_SAMPLE_ROWS, replace=False))
    else:
        pca_sample_indices = np.arange(pca_rows)

    feature_sample = segment_reduction.feature_matrix[pca_sample_indices]
    feature_dense = feature_sample.toarray() if hasattr(feature_sample, "toarray") else np.asarray(feature_sample)
    feature_scaled_for_pca = StandardScaler().fit_transform(feature_dense)
    pca_components = min(SEGMENT_COMPONENTS, feature_scaled_for_pca.shape[0] - 1, feature_scaled_for_pca.shape[1])
    segment_pca_model = PCA(n_components=pca_components, random_state=RANDOM_SEED)
    segment_pca_sample_embeddings = segment_pca_model.fit_transform(feature_scaled_for_pca).astype(np.float32)

    pca_explained = pd.DataFrame(
        {
            "component": np.arange(1, segment_pca_model.n_components_ + 1),
            "explained_variance_ratio": segment_pca_model.explained_variance_ratio_,
            "cumulative_explained_variance_ratio": np.cumsum(segment_pca_model.explained_variance_ratio_),
        }
    )
    display(pca_explained.head(12))
    print(f"PCA diagnostic rows: {len(pca_sample_indices):,}/{pca_rows:,}")

    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].plot(
        pca_explained["component"],
        pca_explained["cumulative_explained_variance_ratio"],
        marker="o",
        linewidth=1.5,
        color="#4C78A8",
    )
    axes[0].set_title("PCA cumulative explained variance")
    axes[0].set_xlabel("PCA component")
    axes[0].set_ylabel("Cumulative explained variance ratio")
    axes[0].set_ylim(0, min(1.0, max(0.1, pca_explained["cumulative_explained_variance_ratio"].max() * 1.1)))

    n_bars = min(15, len(pca_explained))
    axes[1].bar(
        pca_explained["component"].iloc[:n_bars],
        pca_explained["explained_variance_ratio"].iloc[:n_bars],
        color="#F28E2B",
        alpha=0.9,
    )
    axes[1].set_title("Top PCA component variance")
    axes[1].set_xlabel("PCA component")
    axes[1].set_ylabel("Explained variance ratio")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "segment_pca_explained_variance.png", dpi=160)
    plt.show()
else:
    pca_sample_indices = np.array([], dtype=int)
    segment_pca_sample_embeddings = np.empty((0, 0), dtype=np.float32)
    print("PCA diagnostic skipped: not enough rows or selected features.")


### 3.2 Choose HDBSCAN settings

Compare candidate settings on a segment sample. Scale the selected `min_cluster_size` for the final fit and carry over `min_samples`.


In [ ]:
from collections import Counter
from sklearn.metrics import calinski_harabasz_score, davies_bouldin_score, silhouette_score


def _sample_embedding_indices(n_rows: int, max_rows: int, seed: int) -> np.ndarray:
    if n_rows <= max_rows:
        return np.arange(n_rows, dtype=np.int64)
    rng = np.random.default_rng(seed)
    return np.sort(rng.choice(n_rows, size=max_rows, replace=False))


def _candidate_hdbscan_params(n_segments: int) -> list[dict[str, int]]:
    """Build an exploratory HDBSCAN grid scaled to the current segment count."""
    if n_segments <= 0:
        return []

    size_floor = max(8, HDBSCAN_MIN_SAMPLES + 2)
    ratio_candidates = [0.003, 0.004, 0.006, 0.008, 0.010, 0.015, 0.020, 0.030]
    absolute_candidates = [200, HDBSCAN_MIN_CLUSTER_SIZE, 450, 600, 900, 1200, 1800, 2400]
    size_candidates = sorted(
        {max(size_floor, int(round(n_segments * ratio))) for ratio in ratio_candidates}
        | {max(size_floor, value) for value in absolute_candidates}
    )
    size_candidates = [value for value in size_candidates if value < max(size_floor + 1, n_segments // 2)]

    max_size_candidates = max(8, HDBSCAN_MAX_CANDIDATES // 4)
    if len(size_candidates) > max_size_candidates:
        positions = np.linspace(0, len(size_candidates) - 1, max_size_candidates).round().astype(int)
        size_candidates = [size_candidates[index] for index in sorted(set(positions))]

    sample_candidates = sorted({5, HDBSCAN_MIN_SAMPLES, 15, 20})
    params = [
        {"min_cluster_size": int(size), "min_samples": int(samples)}
        for size in size_candidates
        for samples in sample_candidates
        if samples <= size
    ]
    return params[:HDBSCAN_MAX_CANDIDATES]


def _safe_cluster_metric(func, embeddings: np.ndarray, labels: np.ndarray, mask: np.ndarray, **metric_kwargs) -> float:
    unique = np.unique(labels[mask]) if mask.any() else np.array([])
    if len(unique) < 2 or mask.sum() <= len(unique):
        return float("nan")
    try:
        return float(func(embeddings[mask], labels[mask], **metric_kwargs))
    except Exception:
        return float("nan")


def _score_clustering(row: dict[str, float]) -> float:
    cluster_count = row["cluster_count"]
    if cluster_count < 2:
        return -10.0

    silhouette = row["silhouette_non_noise"]
    if not np.isfinite(silhouette):
        silhouette = -0.15

    target_cluster_count = HDBSCAN_TARGET_CLUSTERS
    cluster_count_penalty = 0.06 * abs(cluster_count - target_cluster_count)
    excess_noise_penalty = 0.35 * max(0.0, row["noise_ratio"] - 0.30)
    dominance_penalty = 0.25 * max(0.0, row["largest_cluster_share"] - 0.45)
    tiny_cluster_penalty = 0.15 * row["tiny_cluster_ratio"]

    return float(
        silhouette
        + 0.30 * row["coverage"]
        + 0.15 * row["mean_probability_non_noise"]
        - cluster_count_penalty
        - excess_noise_penalty
        - dominance_penalty
        - tiny_cluster_penalty
    )


def _evaluate_labels(
    embeddings: np.ndarray,
    labels: np.ndarray,
    probabilities: np.ndarray,
    method: str,
    min_cluster_size: int | None = None,
    min_samples: int | None = None,
) -> dict[str, float | int | str]:
    labels = labels.astype(int)
    non_noise = labels >= 0
    cluster_ids = sorted(int(value) for value in np.unique(labels[non_noise]))
    counts = Counter(int(value) for value in labels[non_noise])
    largest_cluster_share = max(counts.values()) / len(labels) if counts else 0.0
    tiny_cluster_count = sum(1 for value in counts.values() if value < max(10, int(0.02 * len(labels))))
    tiny_cluster_ratio = tiny_cluster_count / max(1, len(cluster_ids))

    row = {
        "method": method,
        "min_cluster_size": min_cluster_size if min_cluster_size is not None else -1,
        "min_samples": min_samples if min_samples is not None else -1,
        "segment_count": int(len(labels)),
        "cluster_count": int(len(cluster_ids)),
        "noise_count": int((labels == -1).sum()),
        "noise_ratio": float((labels == -1).mean()) if len(labels) else 0.0,
        "coverage": float(non_noise.mean()) if len(labels) else 0.0,
        "largest_cluster_share": float(largest_cluster_share),
        "tiny_cluster_ratio": float(tiny_cluster_ratio),
        "mean_probability_non_noise": float(np.nanmean(probabilities[non_noise])) if non_noise.any() else 0.0,
        "silhouette_non_noise": _safe_cluster_metric(
            silhouette_score,
            embeddings,
            labels,
            non_noise,
            sample_size=int(min(HDBSCAN_SILHOUETTE_SAMPLE_ROWS, non_noise.sum())),
            random_state=RANDOM_SEED,
        ),
        "calinski_harabasz_non_noise": _safe_cluster_metric(calinski_harabasz_score, embeddings, labels, non_noise),
        "davies_bouldin_non_noise": _safe_cluster_metric(davies_bouldin_score, embeddings, labels, non_noise),
    }
    row["selection_score"] = _score_clustering(row)
    return row


def _predict_hdbscan_batches(clusterer, embeddings: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    import hdbscan

    labels = np.empty(len(embeddings), dtype=np.int64)
    probabilities = np.empty(len(embeddings), dtype=np.float32)
    for start in range(0, len(embeddings), HDBSCAN_PREDICT_BATCH_ROWS):
        end = min(start + HDBSCAN_PREDICT_BATCH_ROWS, len(embeddings))
        batch_labels, batch_probabilities = hdbscan.approximate_predict(clusterer, embeddings[start:end])
        labels[start:end] = batch_labels.astype(np.int64)
        probabilities[start:end] = batch_probabilities.astype(np.float32)
    return labels, probabilities


def run_hdbscan_candidate_search(embeddings: np.ndarray) -> tuple[pd.DataFrame, dict[str, object]]:
    """Evaluate HDBSCAN candidates and return selected full assignments."""
    if len(embeddings) == 0:
        empty = pd.DataFrame()
        return empty, {"labels": np.array([], dtype=int), "probabilities": np.array([], dtype=float), "method": "empty"}

    search_indices = _sample_embedding_indices(len(embeddings), HDBSCAN_SEARCH_SAMPLE_ROWS, RANDOM_SEED + 17)
    fit_indices = _sample_embedding_indices(len(embeddings), HDBSCAN_FIT_SAMPLE_ROWS, RANDOM_SEED + 23)
    search_embeddings = embeddings[search_indices]
    fit_embeddings = embeddings[fit_indices]
    print(f"HDBSCAN search sample: {len(search_indices):,}/{len(embeddings):,} segments")
    print(f"HDBSCAN fit sample: {len(fit_indices):,}/{len(embeddings):,} segments")

    try:
        import hdbscan
    except Exception as exc:
        from sklearn.cluster import MiniBatchKMeans
        k = min(HDBSCAN_TARGET_CLUSTERS, max(2, len(embeddings) // 80))
        model = MiniBatchKMeans(n_clusters=k, n_init=10, random_state=RANDOM_SEED, batch_size=4096)
        labels = model.fit_predict(embeddings).astype(int)
        probabilities = np.ones(len(labels), dtype=float)
        row = _evaluate_labels(embeddings, labels, probabilities, f"minibatch_kmeans_fallback(k={k}, reason={type(exc).__name__})")
        return pd.DataFrame([row]), {"labels": labels, "probabilities": probabilities, "method": row["method"]}

    candidates = []
    param_list = _candidate_hdbscan_params(len(search_embeddings))
    for candidate_index, params in enumerate(param_list, start=1):
        # core_dist_n_jobs=1 keeps the core-distance stage in-process. The
        # default (4) spawns a joblib/loky worker pool, which crashes with
        # BrokenProcessPool on Windows under a memory-heavy Jupyter kernel
        # (hdbscan 0.8.44 + joblib 1.5.x).
        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=params["min_cluster_size"],
            min_samples=params["min_samples"],
            metric="euclidean",
            cluster_selection_method="eom",
            prediction_data=False,
            core_dist_n_jobs=1,
        )
        try:
            labels = clusterer.fit_predict(search_embeddings).astype(int)
        except Exception as exc:
            print(f"[{candidate_index}/{len(param_list)}] skipping {params}: {type(exc).__name__}: {exc}")
            continue
        probabilities = getattr(clusterer, "probabilities_", np.ones(len(labels), dtype=float)).astype(float)
        method = f"hdbscan(min_cluster_size={params['min_cluster_size']}, min_samples={params['min_samples']})"
        row = _evaluate_labels(
            search_embeddings,
            labels,
            probabilities,
            method,
            min_cluster_size=params["min_cluster_size"],
            min_samples=params["min_samples"],
        )
        candidates.append(row)
        print(
            f"[{candidate_index}/{len(param_list)}] {method}: "
            f"clusters={row['cluster_count']}, noise={row['noise_ratio']:.1%}, score={row['selection_score']:.3f}",
            flush=True,
        )

    quality_table = pd.DataFrame(candidates).sort_values("selection_score", ascending=False).reset_index(drop=True)
    if quality_table.empty:
        return quality_table, {"labels": np.full(len(embeddings), -1, dtype=int), "probabilities": np.zeros(len(embeddings)), "method": "hdbscan_no_candidates"}

    best = quality_table.iloc[0].to_dict()
    best_params = {"min_cluster_size": int(best["min_cluster_size"]), "min_samples": int(best["min_samples"])}

    # min_cluster_size is an absolute member count, so the value selected on
    # the search sample encodes a cluster-share floor. Transfer that share to
    # the final fit size; min_samples is a density k and transfers as-is.
    if len(fit_indices) != len(search_indices):
        selected_share = best_params["min_cluster_size"] / max(1, len(search_indices))
        scaled_min_cluster_size = max(8, best_params["min_samples"] + 1, int(round(selected_share * len(fit_indices))))
        print(
            f"Rescaling min_cluster_size {best_params['min_cluster_size']} "
            f"(share {selected_share:.3%} of {len(search_indices):,} search rows) "
            f"-> {scaled_min_cluster_size} for {len(fit_indices):,} fit rows"
        )
        best_params["min_cluster_size"] = scaled_min_cluster_size

    final_clusterer = hdbscan.HDBSCAN(
        min_cluster_size=best_params["min_cluster_size"],
        min_samples=best_params["min_samples"],
        metric="euclidean",
        cluster_selection_method="eom",
        prediction_data=True,
        core_dist_n_jobs=1,
    )
    fit_labels = final_clusterer.fit_predict(fit_embeddings).astype(int)
    fit_probabilities = getattr(final_clusterer, "probabilities_", np.ones(len(fit_labels), dtype=float)).astype(float)
    method = f"hdbscan_sample_fit(min_cluster_size={best_params['min_cluster_size']}, min_samples={best_params['min_samples']}, fit_rows={len(fit_indices)})"

    if len(fit_indices) == len(embeddings):
        labels = fit_labels
        probabilities = fit_probabilities
    elif len(set(fit_labels)) - (1 if -1 in fit_labels else 0) >= 1:
        labels, probabilities = _predict_hdbscan_batches(final_clusterer, embeddings)
    else:
        labels = np.full(len(embeddings), -1, dtype=np.int64)
        probabilities = np.zeros(len(embeddings), dtype=np.float32)

    return quality_table, {
        "labels": labels.astype(int),
        "probabilities": probabilities.astype(float),
        "clusterer": final_clusterer,
        "method": method,
        "params": best_params,
        "search_indices": search_indices,
        "fit_indices": fit_indices,
        "fit_labels": fit_labels,
    }


cluster_quality_table, best_cluster_result = run_hdbscan_candidate_search(segment_embeddings)
cluster_labels = best_cluster_result["labels"]
cluster_probabilities = best_cluster_result["probabilities"]
cluster_method = str(best_cluster_result["method"])

print(f"Selected cluster method: {cluster_method}")
if not cluster_quality_table.empty:
    display(cluster_quality_table.head(15))


### 3.3 Assign segments to clusters

Save the cluster ID and membership probability for each segment. Noise retains the ID `-1`.


In [ ]:
clustered_segment_table = segment_table.copy()
if len(cluster_labels):
    clustered_segment_table["cluster_id"] = cluster_labels.astype(int)
    clustered_segment_table["cluster_probability"] = cluster_probabilities.astype(float)
else:
    clustered_segment_table["cluster_id"] = pd.Series(dtype=int)
    clustered_segment_table["cluster_probability"] = pd.Series(dtype=float)
clustered_segment_table["is_noise"] = clustered_segment_table["cluster_id"] == -1

cluster_label_counts = clustered_segment_table["cluster_id"].value_counts().sort_index() if len(clustered_segment_table) else pd.Series(dtype=int)
cluster_overview_rows = []
for cluster_id, count in cluster_label_counts.items():
    subset = clustered_segment_table.loc[clustered_segment_table["cluster_id"] == cluster_id]
    cluster_overview_rows.append(
        {
            "cluster_id": int(cluster_id),
            "segment_count": int(count),
            "share": float(count / len(clustered_segment_table)),
            "mean_probability": float(subset["cluster_probability"].mean()),
            "median_activity_count": float(subset["activity_count"].median()),
            "median_duration_seconds": float(subset["duration_seconds"].median()),
            "top_close_reason": str(subset["close_reason"].mode().iloc[0]) if not subset.empty else "",
            "dominant_activity": str(subset["dominant_activity"].mode().iloc[0]) if not subset.empty else "",
        }
    )
cluster_overview_table = pd.DataFrame(cluster_overview_rows).sort_values("cluster_id")

print("Cluster size overview (-1 is HDBSCAN noise):")
display(cluster_overview_table)


def representative_segments_for_clusters(
    embeddings: np.ndarray,
    labels: np.ndarray,
    probabilities: np.ndarray,
    table: pd.DataFrame,
    top_n: int = 5,
) -> pd.DataFrame:
    """Pick segments nearest to each cluster centroid as compact examples."""
    rows = []
    if len(labels) == 0:
        return pd.DataFrame(rows)
    for cluster_id in sorted(int(value) for value in np.unique(labels) if int(value) >= 0):
        indices = np.where(labels == cluster_id)[0]
        if len(indices) == 0:
            continue
        centroid = embeddings[indices].mean(axis=0)
        distances = np.linalg.norm(embeddings[indices] - centroid, axis=1)
        order = np.lexsort((-probabilities[indices], distances))[:top_n]
        for rank, local_index in enumerate(order, start=1):
            row_index = int(indices[local_index])
            segment_row = table.iloc[row_index]
            trace = str(segment_row.get("trace", ""))
            rows.append(
                {
                    "cluster_id": cluster_id,
                    "rank": rank,
                    "segment_id": segment_row["segment_id"],
                    "session_index": int(segment_row["session_index"]),
                    "activity_count": int(segment_row["activity_count"]),
                    "duration_seconds": float(segment_row["duration_seconds"]),
                    "cluster_probability": float(probabilities[row_index]),
                    "distance_to_centroid": float(distances[local_index]),
                    "trace_preview": trace[:220] + ("..." if len(trace) > 220 else ""),
                }
            )
    return pd.DataFrame(rows)


cluster_representative_table = representative_segments_for_clusters(
    segment_embeddings,
    cluster_labels,
    cluster_probabilities,
    clustered_segment_table,
    top_n=4,
)

if not cluster_representative_table.empty:
    display(cluster_representative_table)


### 3.4 Compare cluster features

Feature lift shows which features are more common in a cluster than across all segments. Activity shares help interpret its process map.


In [ ]:
feature_rankings = cluster_feature_rankings(
    segment_reduction.feature_matrix,
    segment_reduction.feature_names,
    cluster_labels,
    top_n=15,
)

ranking_rows = []
for cluster_id, features in feature_rankings.items():
    for rank, (feature, lift) in enumerate(features, start=1):
        ranking_rows.append({"cluster_id": int(cluster_id), "rank": rank, "feature": feature, "lift": lift})
feature_ranking_table = pd.DataFrame(ranking_rows, columns=["cluster_id", "rank", "feature", "lift"])

if not feature_ranking_table.empty:
    display(feature_ranking_table.head(100))

case_by_segment_id = {segment["segment_id"]: dict(segment, case_id=segment["segment_id"]) for segment in segments}
activity_counter_by_cluster: dict[int, Counter[str]] = {}
for _, row in clustered_segment_table.iterrows():
    cluster_id = int(row["cluster_id"])
    if cluster_id < 0:
        continue
    case = case_by_segment_id.get(row["segment_id"])
    if not case:
        continue
    activity_counter_by_cluster.setdefault(cluster_id, Counter()).update(case.get("activities", []))

top_activity_names = [activity for activity, _ in Counter({}).items()]
all_activity_counts = Counter()
for counter in activity_counter_by_cluster.values():
    all_activity_counts.update(counter)
top_activity_names = [activity for activity, _ in all_activity_counts.most_common(14)]

activity_matrix_rows = []
for cluster_id, counter in sorted(activity_counter_by_cluster.items()):
    total = max(1, sum(counter.values()))
    row = {"cluster_id": cluster_id, "activity_total": total}
    for activity in top_activity_names:
        row[activity] = counter.get(activity, 0) / total
    activity_matrix_rows.append(row)
cluster_activity_matrix = pd.DataFrame(activity_matrix_rows)

if not cluster_activity_matrix.empty:
    display(cluster_activity_matrix)


### 3.5 Cluster diagnostics and visualizations


In [ ]:
if not cluster_quality_table.empty:
    top_quality = cluster_quality_table.head(12).sort_values("selection_score", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    labels_for_plot = [
        f"mcs={int(row.min_cluster_size)}, ms={int(row.min_samples)}"
        for row in top_quality.itertuples()
    ]
    ax.barh(labels_for_plot, top_quality["selection_score"], color="#4C78A8", alpha=0.9)
    ax.set_title("Top HDBSCAN candidate configurations")
    ax.set_xlabel("Selection score")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "cluster_parameter_search.png", dpi=160)
    plt.show()

if len(cluster_labels) and segment_embeddings.shape[1] >= 2:
    unique_labels = sorted(np.unique(cluster_labels))
    color_map = {label: "#BAB0AC" for label in unique_labels}
    palette = plt.cm.tab20(np.linspace(0, 1, max(1, len([label for label in unique_labels if label >= 0]))))
    for color_index, label in enumerate([label for label in unique_labels if label >= 0]):
        color_map[label] = palette[color_index]
    colors = [color_map[int(label)] for label in cluster_labels]
    alpha = np.clip(0.25 + 0.75 * cluster_probabilities, 0.25, 1.0)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        segment_embeddings[:, 0],
        segment_embeddings[:, 1],
        c=colors,
        s=30,
        alpha=alpha,
        linewidth=0,
    )
    ax.set_title("Reduced segment representation, colored by selected cluster")
    ax.set_xlabel("SVD component 1")
    ax.set_ylabel("SVD component 2")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "segment_clusters_svd_scatter.png", dpi=160)
    plt.show()


if len(cluster_labels) and segment_pca_sample_embeddings.shape[1] >= 2 and len(pca_sample_indices):
    sampled_labels = cluster_labels[pca_sample_indices]
    sampled_probabilities = cluster_probabilities[pca_sample_indices]
    unique_labels = sorted(np.unique(sampled_labels))
    color_map = {label: "#BAB0AC" for label in unique_labels}
    palette = plt.cm.tab20(np.linspace(0, 1, max(1, len([label for label in unique_labels if label >= 0]))))
    for color_index, label in enumerate([label for label in unique_labels if label >= 0]):
        color_map[label] = palette[color_index]
    colors = [color_map[int(label)] for label in sampled_labels]
    alpha = np.clip(0.25 + 0.75 * sampled_probabilities, 0.25, 1.0)

    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(
        segment_pca_sample_embeddings[:, 0],
        segment_pca_sample_embeddings[:, 1],
        c=colors,
        s=30,
        alpha=alpha,
        linewidth=0,
    )
    ax.set_title("PCA diagnostic sample, colored by selected cluster")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "segment_clusters_pca_scatter.png", dpi=160)
    plt.show()

if not cluster_overview_table.empty:
    counts = cluster_overview_table.set_index("cluster_id")["segment_count"]
    colors = ["#BAB0AC" if int(idx) == -1 else "#B07AA1" for idx in counts.index]
    fig, ax = plt.subplots(figsize=(8, 4))
    counts.plot(kind="bar", ax=ax, color=colors)
    ax.set_title("Cluster sizes, -1 is HDBSCAN noise")
    ax.set_xlabel("Cluster id")
    ax.set_ylabel("Segment count")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "context_cluster_sizes.png", dpi=160)
    plt.show()

if len(cluster_probabilities):
    fig, ax = plt.subplots(figsize=(8, 4))
    non_noise_probabilities = cluster_probabilities[cluster_labels >= 0]
    if len(non_noise_probabilities):
        ax.hist(non_noise_probabilities, bins=24, color="#59A14F", alpha=0.85)
    ax.set_title("HDBSCAN membership probability for non-noise segments")
    ax.set_xlabel("Membership probability")
    ax.set_ylabel("Segment count")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "cluster_probability_distribution.png", dpi=160)
    plt.show()

if not cluster_activity_matrix.empty and top_activity_names:
    heatmap_data = cluster_activity_matrix.set_index("cluster_id")[top_activity_names]
    fig, ax = plt.subplots(figsize=(max(9, 0.65 * len(top_activity_names)), max(3, 0.5 * len(heatmap_data))))
    image = ax.imshow(heatmap_data.values, aspect="auto", cmap="Blues")
    ax.set_title("Activity share by cluster")
    ax.set_xlabel("Activity")
    ax.set_ylabel("Cluster id")
    ax.set_xticks(np.arange(len(top_activity_names)))
    ax.set_xticklabels(top_activity_names, rotation=45, ha="right")
    ax.set_yticks(np.arange(len(heatmap_data.index)))
    ax.set_yticklabels(heatmap_data.index.astype(str))
    fig.colorbar(image, ax=ax, fraction=0.025, pad=0.02, label="Activity share")
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "cluster_activity_heatmap.png", dpi=160)
    plt.show()



## 4. Fuzzy process mining

Treat each cluster as an event log. Build a process map from transitions between activities and retain the most informative connections.

### 4.1 Build cluster process maps


In [ ]:
from IPython.display import Image, display

pm_cases = [dict(segment, case_id=segment["segment_id"]) for segment in segments]
case_by_id = {case["case_id"]: case for case in pm_cases}

fuzzy_config = FuzzyMinerConfig(
    min_cases=20,
    min_edge_count=2,
    max_nodes=14,
    max_edges=24,
    preserve_top_k=2,
    edge_utility_quantile=0.62,
    significance_weight=0.65,
    correlation_weight=0.35,
    label_top_edges=10,
)
FUZZY_FIGURE_DIR = FIGURE_DIR / "fuzzy_miner"
FUZZY_FIGURE_DIR.mkdir(parents=True, exist_ok=True)
for stale_path in FUZZY_FIGURE_DIR.glob("cluster_*_fuzzy_process.png"):
    stale_path.unlink()
for stale_path in FUZZY_FIGURE_DIR.glob("cluster_*_fuzzy_process_clean.png"):
    stale_path.unlink()
for stale_overview_path in [
    FUZZY_FIGURE_DIR / "all_clusters_fuzzy_process_overview.png",
    FUZZY_FIGURE_DIR / "all_clusters_fuzzy_process_overview_clean.png",
]:
    if stale_overview_path.exists():
        stale_overview_path.unlink()

cluster_summaries = []
fuzzy_graphs = []
fuzzy_diagram_rows = []

if len(cluster_labels):
    for cluster_id in sorted(set(int(label) for label in cluster_labels if int(label) >= 0)):
        segment_ids = clustered_segment_table.loc[
            clustered_segment_table["cluster_id"] == cluster_id,
            "segment_id",
        ].tolist()
        cluster_cases = [case_by_id[segment_id] for segment_id in segment_ids if segment_id in case_by_id]
        if not cluster_cases:
            continue

        summary = summarize_case_group(cluster_cases, pm_config, f"cluster_{cluster_id}")
        subset = clustered_segment_table.loc[clustered_segment_table["cluster_id"] == cluster_id]
        top_features = feature_ranking_table.loc[
            feature_ranking_table["cluster_id"] == cluster_id,
            "feature",
        ].head(5).tolist() if not feature_ranking_table.empty else []

        fuzzy_graph = discover_fuzzy_process(cluster_cases, cluster_id=cluster_id, config=fuzzy_config)
        fuzzy_graphs.append(fuzzy_graph)
        fuzzy_summary = fuzzy_graph["summary"]
        diagram_path = FUZZY_FIGURE_DIR / f"cluster_{cluster_id:02d}_fuzzy_process.png"
        render_fuzzy_process_graph(
            fuzzy_graph,
            diagram_path,
            title=f"Cluster {cluster_id}: {summary['task_subtype']}",
        )
        clean_diagram_path = FUZZY_FIGURE_DIR / f"cluster_{cluster_id:02d}_fuzzy_process_clean.png"

        summary["cluster_probability_mean"] = round(float(subset["cluster_probability"].mean()), 4)
        summary["cluster_probability_median"] = round(float(subset["cluster_probability"].median()), 4)
        summary["median_activity_count"] = round(float(subset["activity_count"].median()), 2)
        summary["median_duration_seconds"] = round(float(subset["duration_seconds"].median()), 2)
        summary["top_lift_features"] = top_features
        summary["fuzzy_retained_edge_count"] = fuzzy_summary["retained_edge_count"]
        summary["fuzzy_edge_count"] = fuzzy_summary["edge_count"]
        summary["fuzzy_retained_edge_coverage"] = round(float(fuzzy_summary["retained_edge_coverage"]), 4)
        summary["fuzzy_trace_variant_count"] = fuzzy_summary["trace_variant_count"]
        summary["fuzzy_diagram_path"] = str(diagram_path)
        summary["fuzzy_clean_diagram_path"] = str(clean_diagram_path)
        render_fuzzy_process_graph(
            fuzzy_graph,
            clean_diagram_path,
            title=f"Cluster {cluster_id} | cases={len(cluster_cases)}",
        )
        cluster_summaries.append(summary)
        fuzzy_diagram_rows.append(
            {
                "cluster_id": cluster_id,
                "case_count": int(len(cluster_cases)),
                "task_subtype": summary["task_subtype"],
                "cluster_probability_mean": summary["cluster_probability_mean"],
                "median_activity_count": summary["median_activity_count"],
                "median_duration_seconds": summary["median_duration_seconds"],
                "diagram_path": str(diagram_path),
                "clean_diagram_path": str(clean_diagram_path),
            }
        )

fuzzy_summary_table, fuzzy_node_table, fuzzy_edge_table = fuzzy_graph_tables(fuzzy_graphs)
fuzzy_diagram_table = pd.DataFrame(fuzzy_diagram_rows)

summary_rows = []
for summary in cluster_summaries:
    summary_rows.append(
        {
            "pattern_id": summary["pattern_id"],
            "case_count": summary["case_count"],
            "activity_count": summary["activity_count"],
            "task_family": summary["task_family"],
            "task_subtype": summary["task_subtype"],
            "taxonomy_confidence": summary["confidence"],
            "cluster_probability_mean": summary["cluster_probability_mean"],
            "median_activity_count": summary["median_activity_count"],
            "median_duration_seconds": summary["median_duration_seconds"],
            "top_activities": ", ".join(item["activity"] for item in summary["top_activities"][:5]),
            "top_lift_features": ", ".join(summary["top_lift_features"][:5]),
            "fuzzy_trace_variant_count": summary["fuzzy_trace_variant_count"],
            "fuzzy_retained_edge_count": summary["fuzzy_retained_edge_count"],
            "fuzzy_edge_count": summary["fuzzy_edge_count"],
            "fuzzy_retained_edge_coverage": summary["fuzzy_retained_edge_coverage"],
            "fuzzy_diagram_path": summary["fuzzy_diagram_path"],
            "fuzzy_clean_diagram_path": summary["fuzzy_clean_diagram_path"],
        }
    )
cluster_summary_table = pd.DataFrame(summary_rows)

if not cluster_summary_table.empty:
    display(cluster_summary_table)

if not fuzzy_summary_table.empty:
    display(
        fuzzy_summary_table.sort_values("case_count", ascending=False).head(20)
    )

pattern_preview_rows = []
for summary in cluster_summaries:
    for item in summary.get("frequent_traces", [])[:3]:
        pattern_preview_rows.append(
            {
                "pattern_id": summary["pattern_id"],
                "kind": "frequent_trace",
                "count": item["count"],
                "pattern": item["trace"],
            }
        )
    for item in summary.get("local_patterns", [])[:3]:
        pattern_preview_rows.append(
            {
                "pattern_id": summary["pattern_id"],
                "kind": f"local_{item['length']}_gram",
                "count": item["count"],
                "pattern": item["pattern"],
            }
        )
cluster_pattern_preview_table = pd.DataFrame(pattern_preview_rows)
if not cluster_pattern_preview_table.empty:
    display(cluster_pattern_preview_table.head(60))

# All cluster diagrams are saved to disk and composed into two overview images:
# one keeps the rule-based task interpretation, and one keeps only cluster metadata.
fuzzy_overview_path = FUZZY_FIGURE_DIR / "all_clusters_fuzzy_process_overview.png"
fuzzy_clean_overview_path = FUZZY_FIGURE_DIR / "all_clusters_fuzzy_process_overview_clean.png"
if not fuzzy_diagram_table.empty:
    render_fuzzy_process_contact_sheet(
        fuzzy_diagram_table,
        fuzzy_overview_path,
        columns=2,
        thumbnail_width=760,
    )
    fuzzy_clean_diagram_table = fuzzy_diagram_table.copy()
    fuzzy_clean_diagram_table["diagram_path"] = fuzzy_clean_diagram_table["clean_diagram_path"]
    render_fuzzy_process_contact_sheet(
        fuzzy_clean_diagram_table,
        fuzzy_clean_overview_path,
        columns=2,
        thumbnail_width=760,
        include_task_subtype=False,
    )
    print(f"Saved fuzzy process diagrams: {len(fuzzy_diagram_table)} clusters in {FUZZY_FIGURE_DIR}")
    print(f"Two-column overview: {fuzzy_overview_path}")
    display(Image(filename=str(fuzzy_overview_path)))
    print(f"Clean two-column overview: {fuzzy_clean_overview_path}")
    display(Image(filename=str(fuzzy_clean_overview_path)))
else:
    print("No non-noise clusters were available for fuzzy process mining.")


### 4.2 Save discovery results

Export segments, cluster summaries, process maps, and the manual taxonomy used by Task Classification.


In [ ]:
def json_default(value):
    if hasattr(value, "isoformat"):
        return value.isoformat()
    if isinstance(value, np.integer):
        return int(value)
    if isinstance(value, np.floating):
        return float(value)
    if isinstance(value, np.ndarray):
        return value.tolist()
    return str(value)

segments_path = OUTPUT_DIR / "context_segments.parquet"
boundaries_path = OUTPUT_DIR / "context_boundaries.parquet"
clusters_path = OUTPUT_DIR / "context_clustered_segments.parquet"
quality_path = OUTPUT_DIR / "context_cluster_quality.parquet"
representatives_path = OUTPUT_DIR / "context_cluster_representatives.parquet"
feature_rankings_path = OUTPUT_DIR / "context_cluster_feature_rankings.parquet"
activity_matrix_path = OUTPUT_DIR / "context_cluster_activity_matrix.parquet"
cluster_summary_path = OUTPUT_DIR / "context_cluster_process_summary.parquet"
pattern_preview_path = OUTPUT_DIR / "context_cluster_pattern_preview.parquet"
fuzzy_summary_path = OUTPUT_DIR / "context_cluster_fuzzy_summary.parquet"
fuzzy_nodes_path = OUTPUT_DIR / "context_cluster_fuzzy_nodes.parquet"
fuzzy_edges_path = OUTPUT_DIR / "context_cluster_fuzzy_edges.parquet"
fuzzy_diagrams_path = OUTPUT_DIR / "context_cluster_fuzzy_diagrams.parquet"
summary_json_path = OUTPUT_DIR / "context_pipeline_summary.json"
patterns_path = OUTPUT_DIR / "context_cluster_process_patterns.json"
taxonomy_path = OUTPUT_DIR / "task_taxonomy.json"

segment_table.to_parquet(segments_path, index=False)
boundary_table.to_parquet(boundaries_path, index=False)
clustered_segment_table.to_parquet(clusters_path, index=False)
cluster_quality_table.to_parquet(quality_path, index=False)
cluster_representative_table.to_parquet(representatives_path, index=False)
feature_ranking_table.to_parquet(feature_rankings_path, index=False)
cluster_activity_matrix.to_parquet(activity_matrix_path, index=False)
cluster_summary_table.to_parquet(cluster_summary_path, index=False)
cluster_pattern_preview_table.to_parquet(pattern_preview_path, index=False)
fuzzy_summary_table.to_parquet(fuzzy_summary_path, index=False)
fuzzy_node_table.to_parquet(fuzzy_nodes_path, index=False)
fuzzy_edge_table.to_parquet(fuzzy_edges_path, index=False)
fuzzy_diagram_table.to_parquet(fuzzy_diagrams_path, index=False)
patterns_path.write_text(json.dumps(cluster_summaries, indent=2, default=json_default), encoding="utf-8")

# Single source of truth for the task taxonomy consumed by notebook 03.
# Cluster ids are run-specific: re-derive RUN_CLUSTER_TO_SUBTASK in
# process_mining/task_taxonomy.py whenever this warns about drift.
from pipeline.process_mining.task_taxonomy import export_taxonomy

observed_cluster_ids = sorted(set(int(label) for label in cluster_labels if int(label) >= 0)) if len(cluster_labels) else []
taxonomy = export_taxonomy(taxonomy_path, observed_cluster_ids=observed_cluster_ids)
print(f"Task taxonomy exported: {taxonomy_path} (schema {taxonomy['schema_version']}, run {taxonomy['run_id']})")

best_quality = cluster_quality_table.iloc[0].to_dict() if not cluster_quality_table.empty else {}
summary = {
    "max_sessions": MAX_SESSIONS,
    "session_stride": SESSION_STRIDE,
    "session_sample_fraction": SESSION_SAMPLE_FRACTION,
    "session_sample_seed": SESSION_SAMPLE_SEED,
    "loaded_sessions": int(loaded_sessions),
    "canonical_events": int(canonical_event_total),
    "canonical_display_sample_rows": int(len(canonical_events)),
    "activity_bursts": int(len(activity_bursts)),
    "activity_selected_features": int(len(activity_reduction.feature_names)),
    "activity_fit_sample_rows": int(ACTIVITY_FIT_SAMPLE_ROWS),
    "feature_transform_batch_rows": int(FEATURE_TRANSFORM_BATCH_ROWS),
    "activity_components": int(activity_reduction.n_components),
    "segments": int(len(segment_table)),
    "candidate_boundaries": int(candidate_boundary_count),
    "accepted_boundaries": int(accepted_boundary_count),
    "boundary_threshold": float(boundary_threshold),
    "diagnostic_boundary_rows": int(diagnostic_boundary_count),
    "segment_selected_features": int(len(segment_reduction.feature_names)),
    "segment_components": int(segment_reduction.n_components),
    "cluster_method": cluster_method,
    "hdbscan_target_clusters": int(HDBSCAN_TARGET_CLUSTERS),
    "hdbscan_max_candidates": int(HDBSCAN_MAX_CANDIDATES),
    "hdbscan_search_sample_rows": int(HDBSCAN_SEARCH_SAMPLE_ROWS),
    "hdbscan_fit_sample_rows": int(HDBSCAN_FIT_SAMPLE_ROWS),
    "hdbscan_predict_batch_rows": int(HDBSCAN_PREDICT_BATCH_ROWS),
    "cluster_candidates_evaluated": int(len(cluster_quality_table)),
    "clusters_excluding_noise": int(len(set(int(label) for label in cluster_labels if int(label) >= 0))) if len(cluster_labels) else 0,
    "noise_segments": int((cluster_labels == -1).sum()) if len(cluster_labels) else 0,
    "noise_ratio": float((cluster_labels == -1).mean()) if len(cluster_labels) else 0.0,
    "mean_cluster_probability_non_noise": float(np.mean(cluster_probabilities[cluster_labels >= 0])) if len(cluster_labels) and (cluster_labels >= 0).any() else 0.0,
    "silhouette_non_noise": float(best_quality.get("silhouette_non_noise", float("nan"))) if best_quality else float("nan"),
    "cluster_selection_score": float(best_quality.get("selection_score", float("nan"))) if best_quality else float("nan"),
    "fuzzy_graph_clusters": int(len(fuzzy_summary_table)),
    "fuzzy_diagrams_rendered": int(len(fuzzy_diagram_table)),
    "fuzzy_overview_path": str(fuzzy_overview_path) if "fuzzy_overview_path" in globals() else "",
    "fuzzy_clean_overview_path": str(fuzzy_clean_overview_path) if "fuzzy_clean_overview_path" in globals() else "",
    "fuzzy_median_retained_edge_coverage": float(fuzzy_summary_table["retained_edge_coverage"].median()) if not fuzzy_summary_table.empty else 0.0,
    "short_fragment_ratio": float(short_segment_ratio),
    "max_length_guard_ratio": float(cap_segment_ratio),
    "top10_session_segment_share": float(top10_session_segment_share),
    "median_segment_activities": float(segment_table["activity_count"].median()) if not segment_table.empty else 0.0,
    "median_segment_duration_seconds": float(segment_table["duration_seconds"].median()) if not segment_table.empty else 0.0,
}
summary_json_path.write_text(json.dumps(summary, indent=2, default=json_default), encoding="utf-8")

print(json.dumps(summary, indent=2, default=json_default))
print("Written files:")
for path in [
    segments_path,
    boundaries_path,
    clusters_path,
    quality_path,
    representatives_path,
    feature_rankings_path,
    activity_matrix_path,
    cluster_summary_path,
    pattern_preview_path,
    fuzzy_summary_path,
    fuzzy_nodes_path,
    fuzzy_edges_path,
    fuzzy_diagrams_path,
    patterns_path,
    taxonomy_path,
    summary_json_path,
]:
    if path.exists():
        print(f"- {path}")





# Export aggregate results; per-segment data stay in the intermediate workspace.
export_discovery_results(OUTPUT_DIR, RESULT_DIR)


## 5. Manual task labels

The table records how the clusters from the paper run were interpreted. The labels draw on process maps, activity shares, and context features.

| Task family | Subtask | Supporting clusters | Manual interpretation |
|---|---|---|---|
| Code Understanding / Investigation | Context / File Navigation | C22 | Short `NAVIGATION_BURST <-> IDE_COMMAND_ACTIVITY` loops with little editing. Lowest membership confidence of all clusters (mean 0.32); borders noise. |
| Code Understanding / Investigation | Symbol / Usage Investigation | C23 | Navigation-dominant traces on source files with `find_usages` signals plus some editing. |
| Code Understanding / Investigation | Search / Query Investigation | C10 | `SEARCH_SESSION`-centered traces with navigation and IDE-command support; querying the codebase rather than following references. |
| Code Understanding / Investigation | Runtime-Aware Investigation | C12 | Navigation-dominant with substantial debugger context (`NAV -> IDE -> DEBUG`); investigation guided by runtime state. |
| Code Modification | Completion-Assisted Editing | C1 | Tight `CODE_COMPLETION <-> EDIT_BURST` alternation; active code writing with completion support. |
| Code Modification | Edit-Completion-Navigation Refinement | C2, C9 | Repeated completion/navigation/edit interaction; C2 completion-led, C9 navigation-led with `find_usages`. |
| Code Modification | Command-Assisted Editing | C3, C6 | Edit-heavy `EDIT -> IDE -> EDIT` loops with strong IDE-command participation; C6 is the core editing loop, C3 the same loop in `.xaml`/UI files. |
| Code Modification | Test-Code Editing / Refinement | C5, C8, C21 | Edit/navigation/completion loops whose distinguishing signal is `file_role=test`; writing or maintaining test code (C21 is navigation-lean). |
| Code Modification | Debug-Guided Edit Repair | C4, C15 | `EDIT_BURST` and `DEBUG_SESSION` repeatedly interact; C4 edit-led, C15 debug-led `DEBUG -> EDIT -> DEBUG` repair loops. |
| Runtime Feedback | Build Feedback / Diagnostics | C11 | `BUILD`-centered feedback connected to navigation, edit, and IDE-command follow-up. |
| Runtime Feedback | Tight Debugger Interaction | C16 | `DEBUG -> DEBUG -> DEBUG` with IDE-command control and very long durations; stepping, inspecting, controlling debugger state. |
| Runtime Feedback | Debug-Edit-Navigate Runtime Investigation | C14 | Broad `DEBUG -> NAV -> DEBUG` loop among debugging, navigation, and editing. |
| Project Maintenance / Version Control | VCS Inspection / Sync Maintenance | C0 | `VCS_ACTIVITY`-dominant status/change-list inspection, typically closing with `VCS_SYNC`. |
| Workspace / Context Management | General IDE Tool / Command Operation | C18, C19 | Pure IDE-command bursts (C18: 99% IDE commands) and command/navigation workspace movement without editing (C19). |
| Workspace / Context Management | Workspace-Assisted Editing / Context Switching | C7, C17, C20 | IDE commands alternating with edits, completions, and navigation; workspace operations supporting active coding or context switching. |
| Workspace / Context Management | Command-Driven Debug Interaction | C13 | IDE-command and debug activity tightly connected (`IDE -> DEBUG -> IDE`); command-driven interaction with a running debug session rather than direct code repair. |

The training notebook reads the mapping exported by Section 4.2 from `pipeline/process_mining/task_taxonomy.py`. Review the evidence and update the mapping if discovery is rerun, since cluster IDs depend on the run.


## 6. Reporting figures

These cells read the files saved in Section 4.2 and write figures to `FIGURE_DIR`.


### 6.1 Pipeline funnel

Counts at each stage, from sessions to the task taxonomy.


In [ ]:
# Reporting figure 6.1: data volume at every stage of the discovery pipeline.
rf1_summary_path = OUTPUT_DIR / "context_pipeline_summary.json"
rf1_taxonomy_path = OUTPUT_DIR / "task_taxonomy.json"
if not (rf1_summary_path.exists() and rf1_taxonomy_path.exists()):
    print("Skipped: pipeline summary / taxonomy export not found. Run sections 1-4 (incl. the 4.2 export cell) first.")
else:
    rf1_summary = json.loads(rf1_summary_path.read_text(encoding="utf-8"))
    rf1_taxonomy = json.loads(rf1_taxonomy_path.read_text(encoding="utf-8"))
    rf1_segment_total = int(rf1_summary["segments"])
    rf1_noise_total = int(rf1_summary["noise_segments"])
    rf1_stages = [
        ("Recorded IDE sessions", int(rf1_summary["loaded_sessions"]), None),
        ("Canonical events", int(rf1_summary["canonical_events"]), "canonicalisation + context enrichment"),
        ("Activity bursts", int(rf1_summary["activity_bursts"]), "burst abstraction"),
        ("Segments", rf1_segment_total, "representation-shift segmentation"),
        (
            "Clustered segments",
            rf1_segment_total - rf1_noise_total,
            f"HDBSCAN — {rf1_noise_total:,} noise segments ({float(rf1_summary['noise_ratio']):.1%}) excluded",
        ),
        ("Behaviour clusters", int(rf1_summary["clusters_excluding_noise"]), "cluster assignment"),
        (
            f"{len(rf1_taxonomy['subtasks'])} subtasks / {len(rf1_taxonomy['task_families'])} task families",
            len(rf1_taxonomy["subtasks"]),
            "manual taxonomy mapping (section 5)",
        ),
    ]

    rf1_widths = [max(np.log10(max(count, 2)), 0.6) for _, count, _ in rf1_stages]
    rf1_max_width = max(rf1_widths)
    fig, ax = plt.subplots(figsize=(11.5, 6.4))
    for rf1_row, ((rf1_label, rf1_count, rf1_step), rf1_width) in enumerate(zip(rf1_stages, rf1_widths)):
        rf1_y = -float(rf1_row)
        ax.barh(rf1_y, rf1_width, left=-rf1_width / 2, height=0.56, color="#4C78A8", alpha=0.92, zorder=3)
        rf1_text = rf1_label if rf1_row == len(rf1_stages) - 1 else f"{rf1_label} — {rf1_count:,}"
        ax.text(rf1_max_width / 2 + 0.35, rf1_y, rf1_text, va="center", ha="left", fontsize=10.5)
        if rf1_step:
            ax.annotate(
                "",
                xy=(0, rf1_y + 0.30),
                xytext=(0, rf1_y + 0.70),
                arrowprops=dict(arrowstyle="-|>", color="#7F7F7F", lw=1.1),
                zorder=2,
            )
            ax.text(
                -rf1_max_width / 2 - 0.35,
                rf1_y + 0.5,
                rf1_step,
                va="center",
                ha="right",
                fontsize=8.5,
                color="#5A5A5A",
                style="italic",
            )
    ax.set_xlim(-rf1_max_width / 2 - 7.8, rf1_max_width / 2 + 7.0)
    ax.set_ylim(-(len(rf1_stages) - 1) - 0.7, 0.7)
    ax.set_axis_off()
    ax.set_title("Task-discovery pipeline: data volume per stage", fontsize=13, pad=10)
    fig.text(0.5, 0.02, "Bar width proportional to log10(count).", ha="center", fontsize=8, color="#7F7F7F")
    fig.tight_layout(rect=(0, 0.04, 1, 1))
    fig.savefig(FIGURE_DIR / "report_pipeline_funnel.png", dpi=160, bbox_inches="tight")
    plt.show()

### 6.2 Example session timeline

Colours show activity families, dashed lines mark segment boundaries, and labels show the mapped subtasks. The horizontal axis is activity order.


In [ ]:
# Reporting figure 6.2: one real session as an activity strip with detected boundaries and mapped subtasks.
import textwrap

from matplotlib.patches import Patch

from pipeline.process_mining.case_segmentation import ACTIVITY_FAMILY

rf2_segments_path = OUTPUT_DIR / "context_clustered_segments.parquet"
rf2_taxonomy_path = OUTPUT_DIR / "task_taxonomy.json"
if not (rf2_segments_path.exists() and rf2_taxonomy_path.exists()):
    print("Skipped: clustered segments / taxonomy export not found. Run sections 1-4 (incl. the 4.2 export cell) first.")
else:
    rf2_activity_colors = {
        "editing": "#4C78A8",
        "navigation": "#F58518",
        "runtime_feedback": "#E45756",
        "vcs": "#B279A2",
        "workspace": "#54A24B",
        "system": "#9D9D9D",
        "other": "#D3D3D3",
    }
    rf2_taxonomy = json.loads(rf2_taxonomy_path.read_text(encoding="utf-8"))
    rf2_cluster_to_subtask = rf2_taxonomy["cluster_to_subtask"]
    rf2_segments = pd.read_parquet(
        rf2_segments_path,
        columns=["session_index", "segment_index", "activity_count", "trace", "cluster_id", "is_noise"],
    )
    rf2_stats = rf2_segments.groupby("session_index").agg(
        segment_total=("segment_index", "size"),
        activity_total=("activity_count", "sum"),
        noise_share=("is_noise", "mean"),
    )
    rf2_mapped = rf2_segments.loc[~rf2_segments["is_noise"]].copy()
    rf2_mapped["subtask"] = rf2_mapped["cluster_id"].astype(int).astype(str).map(rf2_cluster_to_subtask)
    rf2_stats["subtask_variety"] = (
        rf2_mapped.groupby("session_index")["subtask"].nunique().reindex(rf2_stats.index).fillna(0)
    )
    rf2_candidates = rf2_stats.loc[
        rf2_stats["segment_total"].between(8, 15)
        & rf2_stats["activity_total"].between(40, 220)
        & (rf2_stats["noise_share"] <= 0.4)
        & (rf2_stats["subtask_variety"] >= 3)
    ]
    if rf2_candidates.empty:
        print("Skipped: no session matches the deterministic example-selection criteria.")
    else:
        rf2_session = int(rf2_candidates.index.min())
        rf2_rows = rf2_segments.loc[rf2_segments["session_index"] == rf2_session].sort_values("segment_index")
        print(
            f"Selected session_index={rf2_session}: {len(rf2_rows)} segments, "
            f"{int(rf2_rows['activity_count'].sum())} activities."
        )

        fig, ax = plt.subplots(figsize=(15, 5.0))
        rf2_cursor = 0
        rf2_seen_families: list[str] = []
        for rf2_position, (_, rf2_seg) in enumerate(rf2_rows.iterrows()):
            rf2_activities = str(rf2_seg["trace"]).split(" -> ")
            for rf2_offset, rf2_activity in enumerate(rf2_activities):
                rf2_family = ACTIVITY_FAMILY.get(rf2_activity, "other")
                if rf2_family not in rf2_seen_families:
                    rf2_seen_families.append(rf2_family)
                ax.add_patch(
                    plt.Rectangle(
                        (rf2_cursor + rf2_offset, 0.0),
                        1.0,
                        1.0,
                        facecolor=rf2_activity_colors.get(rf2_family, rf2_activity_colors["other"]),
                        edgecolor="white",
                        linewidth=0.4,
                    )
                )
            rf2_start, rf2_end = rf2_cursor, rf2_cursor + len(rf2_activities)
            if rf2_position > 0:
                ax.plot([rf2_start, rf2_start], [-0.06, 1.10], color="#B0413E", linestyle="--", linewidth=1.4, zorder=4)
            rf2_label = (
                "(noise)"
                if bool(rf2_seg["is_noise"])
                else rf2_cluster_to_subtask.get(str(int(rf2_seg["cluster_id"])), "(unmapped)")
            )
            rf2_level = (1.30, 2.10, 2.90)[rf2_position % 3]
            rf2_center = (rf2_start + rf2_end) / 2
            ax.plot([rf2_center, rf2_center], [1.06, rf2_level - 0.08], color="#8C8C8C", linewidth=0.7, zorder=3)
            ax.text(
                rf2_center,
                rf2_level,
                "\n".join(textwrap.wrap(rf2_label, 14)),
                ha="center",
                va="bottom",
                fontsize=7.5,
                color="#7F7F7F" if rf2_label == "(noise)" else "#1F1F1F",
            )
            rf2_cursor = rf2_end
        ax.set_xlim(-0.5, rf2_cursor + 0.5)
        ax.set_ylim(-0.25, 4.15)
        ax.set_yticks([])
        ax.set_xlabel("activity index within session (ordinal, not wall-clock time)")
        ax.set_title(
            f"Session {rf2_session}: activity stream, representation-shift boundaries (red), and mapped subtasks"
        )
        ax.legend(
            handles=[Patch(facecolor=rf2_activity_colors[f], label=f) for f in rf2_seen_families],
            loc="upper left",
            bbox_to_anchor=(0.0, -0.14),
            ncol=max(len(rf2_seen_families), 1),
            frameon=False,
            fontsize=8,
            title="activity family",
            title_fontsize=8,
        )
        fig.tight_layout()
        fig.savefig(FIGURE_DIR / "report_session_timeline.png", dpi=160, bbox_inches="tight")
        plt.show()

### 6.3 Cluster → Subtask → Family mapping

Band width shows the number of segments assigned to each label.


In [ ]:
# Reporting figure 6.3: alluvial mapping from HDBSCAN clusters to subtasks to task families.
from matplotlib.patches import PathPatch
from matplotlib.path import Path as MplPath

rf3_segments_path = OUTPUT_DIR / "context_clustered_segments.parquet"
rf3_taxonomy_path = OUTPUT_DIR / "task_taxonomy.json"
if not (rf3_segments_path.exists() and rf3_taxonomy_path.exists()):
    print("Skipped: clustered segments / taxonomy export not found. Run sections 1-4 (incl. the 4.2 export cell) first.")
else:
    rf3_taxonomy = json.loads(rf3_taxonomy_path.read_text(encoding="utf-8"))
    rf3_families = list(rf3_taxonomy["task_families"])
    rf3_family_palette = ["#4C78A8", "#F58518", "#E45756", "#72B7B2", "#54A24B", "#B279A2"]
    rf3_family_colors = {
        rf3_family: rf3_family_palette[rf3_index % len(rf3_family_palette)]
        for rf3_index, rf3_family in enumerate(rf3_families)
    }
    rf3_subtasks = [rf3_item["subtask"] for rf3_item in rf3_taxonomy["subtasks"]]
    rf3_subtask_family = {rf3_item["subtask"]: rf3_item["task_family"] for rf3_item in rf3_taxonomy["subtasks"]}
    rf3_cluster_to_subtask = {int(rf3_key): rf3_value for rf3_key, rf3_value in rf3_taxonomy["cluster_to_subtask"].items()}

    rf3_seg = pd.read_parquet(rf3_segments_path, columns=["cluster_id", "is_noise"])
    rf3_counts = rf3_seg.loc[~rf3_seg["is_noise"], "cluster_id"].astype(int).value_counts().sort_index()
    rf3_counts = rf3_counts.loc[[rf3_cid for rf3_cid in rf3_counts.index if rf3_cid in rf3_cluster_to_subtask]]
    rf3_noise_count = int(rf3_seg["is_noise"].sum())

    rf3_subtask_counts = {rf3_sub: 0 for rf3_sub in rf3_subtasks}
    for rf3_cid, rf3_count in rf3_counts.items():
        rf3_subtask_counts[rf3_cluster_to_subtask[rf3_cid]] += int(rf3_count)
    rf3_family_counts = {rf3_fam: 0 for rf3_fam in rf3_families}
    for rf3_sub, rf3_count in rf3_subtask_counts.items():
        rf3_family_counts[rf3_subtask_family[rf3_sub]] += rf3_count

    rf3_cluster_order = sorted(
        rf3_counts.index,
        key=lambda rf3_cid: (
            rf3_families.index(rf3_subtask_family[rf3_cluster_to_subtask[rf3_cid]]),
            rf3_subtasks.index(rf3_cluster_to_subtask[rf3_cid]),
            rf3_cid,
        ),
    )
    rf3_total = float(rf3_counts.sum())
    rf3_height = rf3_total * 1.22

    rf3_columns = {}
    for rf3_key, rf3_labels, rf3_values in (
        ("cluster", rf3_cluster_order, rf3_counts.to_dict()),
        ("subtask", rf3_subtasks, rf3_subtask_counts),
        ("family", rf3_families, rf3_family_counts),
    ):
        rf3_gap = (rf3_height - rf3_total) / max(len(rf3_labels) - 1, 1)
        rf3_positions, rf3_cursor = {}, 0.0
        for rf3_label in rf3_labels:
            rf3_value = float(rf3_values[rf3_label])
            rf3_positions[rf3_label] = (rf3_cursor, rf3_cursor + rf3_value)
            rf3_cursor += rf3_value + rf3_gap
        rf3_columns[rf3_key] = rf3_positions

    def rf3_band(rf3_ax, rf3_x0, rf3_x1, rf3_b0, rf3_t0, rf3_b1, rf3_t1, rf3_color):
        rf3_xm = (rf3_x0 + rf3_x1) / 2
        rf3_vertices = [
            (rf3_x0, rf3_t0), (rf3_xm, rf3_t0), (rf3_xm, rf3_t1), (rf3_x1, rf3_t1),
            (rf3_x1, rf3_b1), (rf3_xm, rf3_b1), (rf3_xm, rf3_b0), (rf3_x0, rf3_b0), (rf3_x0, rf3_t0),
        ]
        rf3_codes = [
            MplPath.MOVETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4,
            MplPath.LINETO, MplPath.CURVE4, MplPath.CURVE4, MplPath.CURVE4, MplPath.CLOSEPOLY,
        ]
        rf3_ax.add_patch(PathPatch(MplPath(rf3_vertices, rf3_codes), facecolor=rf3_color, alpha=0.48, linewidth=0))

    fig, ax = plt.subplots(figsize=(13.5, 9))
    rf3_x = {"cluster": 0.06, "subtask": 0.52, "family": 0.94}
    rf3_bar = 0.012

    rf3_offsets = {rf3_sub: rf3_columns["subtask"][rf3_sub][0] for rf3_sub in rf3_subtasks}
    for rf3_cid in rf3_cluster_order:
        rf3_sub = rf3_cluster_to_subtask[rf3_cid]
        rf3_value = float(rf3_counts[rf3_cid])
        rf3_b0, rf3_t0 = rf3_columns["cluster"][rf3_cid]
        rf3_b1 = rf3_offsets[rf3_sub]
        rf3_t1 = rf3_b1 + rf3_value
        rf3_offsets[rf3_sub] = rf3_t1
        rf3_band(ax, rf3_x["cluster"] + rf3_bar, rf3_x["subtask"], rf3_b0, rf3_t0, rf3_b1, rf3_t1,
                 rf3_family_colors[rf3_subtask_family[rf3_sub]])

    rf3_offsets = {rf3_fam: rf3_columns["family"][rf3_fam][0] for rf3_fam in rf3_families}
    for rf3_sub in rf3_subtasks:
        rf3_value = float(rf3_subtask_counts[rf3_sub])
        if rf3_value <= 0:
            continue
        rf3_fam = rf3_subtask_family[rf3_sub]
        rf3_b0, rf3_t0 = rf3_columns["subtask"][rf3_sub]
        rf3_b1 = rf3_offsets[rf3_fam]
        rf3_t1 = rf3_b1 + rf3_value
        rf3_offsets[rf3_fam] = rf3_t1
        rf3_band(ax, rf3_x["subtask"] + rf3_bar, rf3_x["family"], rf3_b0, rf3_t0, rf3_b1, rf3_t1,
                 rf3_family_colors[rf3_fam])

    for rf3_cid in rf3_cluster_order:
        rf3_b, rf3_t = rf3_columns["cluster"][rf3_cid]
        rf3_sub = rf3_cluster_to_subtask[rf3_cid]
        ax.add_patch(plt.Rectangle((rf3_x["cluster"], rf3_b), rf3_bar, rf3_t - rf3_b,
                                   facecolor=rf3_family_colors[rf3_subtask_family[rf3_sub]], edgecolor="none"))
        ax.text(rf3_x["cluster"] - 0.008, (rf3_b + rf3_t) / 2, f"C{rf3_cid:02d}", ha="right", va="center", fontsize=7.5)
    for rf3_sub in rf3_subtasks:
        rf3_b, rf3_t = rf3_columns["subtask"][rf3_sub]
        ax.add_patch(plt.Rectangle((rf3_x["subtask"], rf3_b), rf3_bar, rf3_t - rf3_b,
                                   facecolor=rf3_family_colors[rf3_subtask_family[rf3_sub]], edgecolor="none"))
        ax.text(rf3_x["subtask"] + rf3_bar / 2, (rf3_b + rf3_t) / 2,
                f"{rf3_sub}  ({rf3_subtask_counts[rf3_sub]:,})",
                ha="center", va="center", fontsize=7.6,
                bbox=dict(facecolor="white", alpha=0.8, edgecolor="none", pad=1.2))
    for rf3_fam in rf3_families:
        rf3_b, rf3_t = rf3_columns["family"][rf3_fam]
        ax.add_patch(plt.Rectangle((rf3_x["family"], rf3_b), rf3_bar, rf3_t - rf3_b,
                                   facecolor=rf3_family_colors[rf3_fam], edgecolor="none"))
        ax.text(rf3_x["family"] + rf3_bar + 0.010, (rf3_b + rf3_t) / 2,
                "\n".join([rf3_fam, f"({rf3_family_counts[rf3_fam]:,} segments)"]),
                ha="left", va="center", fontsize=8.5)

    for rf3_key, rf3_header, rf3_align in (
        ("cluster", "HDBSCAN clusters", "center"),
        ("subtask", "Subtasks", "center"),
        ("family", "Task families", "left"),
    ):
        ax.text(rf3_x[rf3_key] + rf3_bar / 2, -rf3_height * 0.035, rf3_header,
                ha=rf3_align, va="center", fontsize=10.5, fontweight="bold")
    ax.text(rf3_x["cluster"] - 0.008, rf3_height * 1.045,
            f"{rf3_noise_count:,} noise segments ({rf3_noise_count / len(rf3_seg):.1%}) are excluded from this mapping",
            ha="left", va="center", fontsize=8, color="#7F7F7F")

    ax.set_xlim(0.0, 1.30)
    ax.set_ylim(-rf3_height * 0.07, rf3_height * 1.07)
    ax.invert_yaxis()
    ax.set_axis_off()
    ax.set_title("From unsupervised clusters to the manual task taxonomy (band width = segment count)",
                 fontsize=12.5, pad=12)
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "report_cluster_taxonomy_sankey.png", dpi=160, bbox_inches="tight")
    plt.show()

### 6.4 Activity shares by task family

Cluster activity shares, ordered by the manual taxonomy.


In [ ]:
# Reporting figure 6.4: activity-share heatmap with rows ordered and labelled by the manual taxonomy.
from matplotlib.colors import to_rgb

rf4_matrix_path = OUTPUT_DIR / "context_cluster_activity_matrix.parquet"
rf4_taxonomy_path = OUTPUT_DIR / "task_taxonomy.json"
if not (rf4_matrix_path.exists() and rf4_taxonomy_path.exists()):
    print("Skipped: cluster activity matrix / taxonomy export not found. Run sections 1-4 (incl. the 4.2 export cell) first.")
else:
    rf4_taxonomy = json.loads(rf4_taxonomy_path.read_text(encoding="utf-8"))
    rf4_families = list(rf4_taxonomy["task_families"])
    rf4_family_palette = ["#4C78A8", "#F58518", "#E45756", "#72B7B2", "#54A24B", "#B279A2"]
    rf4_family_colors = {
        rf4_family: rf4_family_palette[rf4_index % len(rf4_family_palette)]
        for rf4_index, rf4_family in enumerate(rf4_families)
    }
    rf4_subtasks = [rf4_item["subtask"] for rf4_item in rf4_taxonomy["subtasks"]]
    rf4_subtask_family = {rf4_item["subtask"]: rf4_item["task_family"] for rf4_item in rf4_taxonomy["subtasks"]}
    rf4_cluster_to_subtask = {int(rf4_key): rf4_value for rf4_key, rf4_value in rf4_taxonomy["cluster_to_subtask"].items()}

    rf4_matrix = pd.read_parquet(rf4_matrix_path)
    rf4_matrix = rf4_matrix.set_index(rf4_matrix["cluster_id"].astype(int))
    rf4_share_columns = [rf4_col for rf4_col in rf4_matrix.columns if rf4_col not in ("cluster_id", "activity_total")]
    rf4_rows = [rf4_cid for rf4_cid in rf4_matrix.index if rf4_cid in rf4_cluster_to_subtask]
    rf4_rows = sorted(
        rf4_rows,
        key=lambda rf4_cid: (
            rf4_families.index(rf4_subtask_family[rf4_cluster_to_subtask[rf4_cid]]),
            rf4_subtasks.index(rf4_cluster_to_subtask[rf4_cid]),
            rf4_cid,
        ),
    )
    rf4_column_order = list(rf4_matrix.loc[rf4_rows, rf4_share_columns].mean().sort_values(ascending=False).index)
    rf4_values = rf4_matrix.loc[rf4_rows, rf4_column_order].to_numpy(dtype=float)
    rf4_labels = [f"C{rf4_cid:02d} · {rf4_cluster_to_subtask[rf4_cid]}" for rf4_cid in rf4_rows]
    rf4_row_families = [rf4_subtask_family[rf4_cluster_to_subtask[rf4_cid]] for rf4_cid in rf4_rows]
    rf4_band = np.array([[to_rgb(rf4_family_colors[rf4_fam])] for rf4_fam in rf4_row_families])

    fig, (rf4_band_ax, rf4_ax) = plt.subplots(
        1, 2, figsize=(13, 8.6), gridspec_kw={"width_ratios": [0.03, 1], "wspace": 0.015}
    )
    rf4_band_ax.imshow(rf4_band, aspect="auto")
    rf4_band_ax.set_xticks([])
    rf4_band_ax.set_yticks(range(len(rf4_labels)))
    rf4_band_ax.set_yticklabels(rf4_labels, fontsize=8.2)
    rf4_band_ax.tick_params(length=0)
    for rf4_spine in rf4_band_ax.spines.values():
        rf4_spine.set_visible(False)

    rf4_image = rf4_ax.imshow(rf4_values, aspect="auto", cmap="Blues", vmin=0.0)
    rf4_ax.set_yticks([])
    rf4_ax.set_xticks(range(len(rf4_column_order)))
    rf4_ax.set_xticklabels(rf4_column_order, rotation=40, ha="right", fontsize=8)
    rf4_threshold = 0.55 * float(rf4_values.max())
    for rf4_r in range(rf4_values.shape[0]):
        for rf4_c in range(rf4_values.shape[1]):
            rf4_value = rf4_values[rf4_r, rf4_c]
            if rf4_value >= 0.10:
                rf4_ax.text(
                    rf4_c, rf4_r, f"{rf4_value:.2f}",
                    ha="center", va="center", fontsize=6.3,
                    color="white" if rf4_value >= rf4_threshold else "#274769",
                )
    for rf4_i in range(1, len(rf4_rows)):
        if rf4_row_families[rf4_i] != rf4_row_families[rf4_i - 1]:
            rf4_ax.axhline(rf4_i - 0.5, color="#2F2F2F", linewidth=1.0)
            rf4_band_ax.axhline(rf4_i - 0.5, color="white", linewidth=1.6)
    fig.colorbar(rf4_image, ax=rf4_ax, fraction=0.035, pad=0.02, label="activity share within cluster")
    rf4_ax.set_title("Cluster activity profiles ordered by the manual task taxonomy", fontsize=12.5, pad=10)
    fig.savefig(FIGURE_DIR / "report_taxonomy_activity_heatmap.png", dpi=160, bbox_inches="tight")
    plt.show()

### 6.5 Subtask transitions

Transition probabilities between consecutive labelled segments in a session.


In [ ]:
# Reporting figure 6.5: how developers move between subtasks across consecutive segments.
rf5_segments_path = OUTPUT_DIR / "context_clustered_segments.parquet"
rf5_taxonomy_path = OUTPUT_DIR / "task_taxonomy.json"
if not (rf5_segments_path.exists() and rf5_taxonomy_path.exists()):
    print("Skipped: clustered segments / taxonomy export not found. Run sections 1-4 (incl. the 4.2 export cell) first.")
else:
    rf5_taxonomy = json.loads(rf5_taxonomy_path.read_text(encoding="utf-8"))
    rf5_subtasks = [rf5_item["subtask"] for rf5_item in rf5_taxonomy["subtasks"]]
    rf5_subtask_family = {rf5_item["subtask"]: rf5_item["task_family"] for rf5_item in rf5_taxonomy["subtasks"]}

    rf5_seg = pd.read_parquet(
        rf5_segments_path, columns=["session_index", "segment_index", "cluster_id", "is_noise"]
    ).sort_values(["session_index", "segment_index"])
    rf5_seg["subtask"] = rf5_seg["cluster_id"].astype(int).astype(str).map(rf5_taxonomy["cluster_to_subtask"])
    rf5_next = rf5_seg.groupby("session_index", sort=False).shift(-1)
    rf5_mask = (~rf5_seg["is_noise"]) & (rf5_next["is_noise"] == False)  # noqa: E712 - NaN-safe comparison
    rf5_pairs = pd.DataFrame(
        {"source": rf5_seg.loc[rf5_mask, "subtask"], "target": rf5_next.loc[rf5_mask, "subtask"]}
    ).dropna()
    print(f"Transitions used: {len(rf5_pairs):,} consecutive non-noise segment pairs.")

    rf5_matrix = (
        pd.crosstab(rf5_pairs["source"], rf5_pairs["target"])
        .reindex(index=rf5_subtasks, columns=rf5_subtasks, fill_value=0)
        .astype(float)
    )
    rf5_probabilities = rf5_matrix.div(rf5_matrix.sum(axis=1).replace(0.0, np.nan), axis=0)
    rf5_self_mean = float(np.nanmean(np.diag(rf5_probabilities.to_numpy())))

    fig, ax = plt.subplots(figsize=(11.5, 9.5))
    rf5_values = rf5_probabilities.to_numpy()
    rf5_image = ax.imshow(rf5_values, aspect="auto", cmap="Blues", vmin=0.0)
    ax.set_xticks(range(len(rf5_subtasks)))
    ax.set_xticklabels(rf5_subtasks, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(rf5_subtasks)))
    ax.set_yticklabels(rf5_subtasks, fontsize=8)
    rf5_threshold = 0.55 * float(np.nanmax(rf5_values))
    for rf5_r in range(len(rf5_subtasks)):
        for rf5_c in range(len(rf5_subtasks)):
            rf5_value = rf5_values[rf5_r, rf5_c]
            if np.isfinite(rf5_value) and rf5_value >= 0.05:
                ax.text(
                    rf5_c, rf5_r, f"{rf5_value:.2f}",
                    ha="center", va="center", fontsize=6.5,
                    color="white" if rf5_value >= rf5_threshold else "#274769",
                )
    rf5_row_families = [rf5_subtask_family[rf5_sub] for rf5_sub in rf5_subtasks]
    for rf5_i in range(1, len(rf5_subtasks)):
        if rf5_row_families[rf5_i] != rf5_row_families[rf5_i - 1]:
            ax.axhline(rf5_i - 0.5, color="#2F2F2F", linewidth=1.0)
            ax.axvline(rf5_i - 0.5, color="#2F2F2F", linewidth=1.0)
    fig.colorbar(rf5_image, ax=ax, fraction=0.04, pad=0.02, label="P(next subtask | current subtask)")
    ax.set_xlabel("next subtask")
    ax.set_ylabel("current subtask")
    ax.set_title(
        "Subtask transition probabilities (row-normalised; consecutive non-noise segments within a session)\n"
        f"mean self-transition probability = {rf5_self_mean:.2f}",
        fontsize=11.5,
        pad=10,
    )
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / "report_subtask_transition_matrix.png", dpi=160, bbox_inches="tight")
    plt.show()